# Import libraries and set configs

In [1]:
import json
import pandas as pd


class CFG:
    n_repeats = 1
    n_folds = 4

# Load the train data

In [2]:
train_df = pd.read_pickle("data/train_df.pkl")

# Feature selection

### Select features with BORUTA feature importance

In [3]:
from utils.feature_selection_utils import boruta_selction

features = [
    c
    for c in train_df.columns
    if c
    not in [
        "time",
        "target",
        "ticker",
        "pattern",
        "ttype",
        "weight",
        "max_price_deviation",
        "min_price_deviation",
        "close_time",
        "first_price",
        "last_price",
    ]
]

params = {
    "boosting_type": "gbdt",
    "n_estimators": 1000,
    "learning_rate": 0.02,
    "max_depth": 6,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "verbosity": -1,
    "importance_type": "gain",
    "objective": "binary",
    "metric": "average_precison",
    "verbose": -1,
}

# boruta_df_ = boruta_selction(train_df, features, params) #!

/home/alex/Repos/sigbot/.venv/lib/python3.12/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/alex/Repos/sigbot/.venv/lib/python3.12/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### Select features with permutation importance and GBM feature importance

In [4]:
from utils.feature_selection_utils import lgbm_tuning

# load the list of Bybit tickers
with open("model/bybit_tickers.json", "r") as f:
    bybit_tickers = json.load(f)

# perm_df_, feature_importances_, outer_cv_score = lgbm_tuning( #!
#     train_df, 
#     features, 
#     params, 
#     bybit_tickers, 
#     n_folds=CFG.n_folds, 
#     n_repeats=CFG.n_repeats, 
#     permut=True,
# )

### RFE feature selection

In [5]:
from utils.feature_selection_utils import rfe_selection

rfe_df_ = rfe_selection(train_df, features)

Fitting estimator with 1571 features.
Fitting estimator with 1532 features.
Fitting estimator with 1493 features.
Fitting estimator with 1454 features.
Fitting estimator with 1415 features.
Fitting estimator with 1376 features.
Fitting estimator with 1337 features.
Fitting estimator with 1298 features.
Fitting estimator with 1259 features.
Fitting estimator with 1220 features.
Fitting estimator with 1181 features.
Fitting estimator with 1142 features.
Fitting estimator with 1103 features.
Fitting estimator with 1064 features.
Fitting estimator with 1025 features.
Fitting estimator with 986 features.
Fitting estimator with 947 features.
Fitting estimator with 908 features.
Fitting estimator with 869 features.
Fitting estimator with 830 features.
Fitting estimator with 791 features.
Fitting estimator with 752 features.
Fitting estimator with 713 features.
Fitting estimator with 674 features.
Fitting estimator with 635 features.
Fitting estimator with 596 features.
Fitting estimator with 

### Combine importances and save them

In [6]:
boruta_df_ = pd.read_csv("model/features/boruta_importances.csv") #!

In [7]:
perm_df_= pd.read_csv("model/features/permutation_importances.csv") #!

In [8]:
rfe_df_

,Feature,importance
0,atr,14
1,atr_prev_100,22
2,atr_prev_104,35
3,atr_prev_108,36
4,atr_prev_112,24
...,...,...
1566,volume_prev_84,35
1567,volume_prev_88,16
1568,volume_prev_92,16
1569,volume_prev_96,18


In [9]:
feature_importances_ = pd.read_csv("model/features/lgbm_feature_importances.csv") #!

In [10]:
boruta_df_["rank"] = boruta_df_["importance"].rank()
perm_df_["rank"] = perm_df_["importance"].rank(ascending=False)
rfe_df_["rank"] = rfe_df_["importance"]
feature_importances_["rank"] = feature_importances_["Value"].rank(ascending=False)

fi = pd.concat([
    perm_df_[["Feature","rank"]], 
    feature_importances_[["Feature","rank"]], 
    rfe_df_[["Feature","rank"]],
    boruta_df_[["Feature","rank"]],
    # pd.DataFrame({"Feature": ["weekday"], "rank": [5000]}),
                ])
fi = fi.groupby("Feature")["rank"].sum().reset_index()
for feature in ["weekday", "funding_rate", "macdhist_prev_4"]:
    fi.loc[fi["Feature"] == feature, "rank"] = fi.loc[fi["Feature"] == feature, "rank"].values[0] / 100
fi = fi.sort_values("rank").reset_index(drop=True)
fi.to_csv("model/features/feature_importance.csv", index=False)
fi

,Feature,rank
0,weekday,10.05
1,funding_rate,21.34
2,macdhist_prev_4,26.06
3,fng_value_prev_24,31.50
4,btcd_volume_prev_144,32.50
...,...,...
1566,btcdom_high_prev_36,3711.00
1567,btcdom_close_prev_172,3716.00
1568,btcdom_low_prev_32,3717.00
1569,btcdom_close_prev_32,3729.00
